# CSI supervised classification: separate experiment

Binary presence and room-specific count categories. Uses the same dataset
contract as csi_modeling.ipynb, but never invents occupancy labels. The supplied
capture has no ground truth, so model comparisons are skipped until annotations
and independent training sessions are added.

Use the ESP-IDF Python kernel and `requirements-modeling.txt`. All feature
selection, preprocessing, estimator arrays, and scoring hooks are visible.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA

relative = Path('drafts/modeling_and_analysis')
search = [candidate for parent in (Path.cwd(), *Path.cwd().parents)
          for candidate in (parent, parent/relative)]
EXAMPLE_DIR = next((p for p in search if (p/'csi_dataset.py').is_file()), None)
if EXAMPLE_DIR is None:
    raise FileNotFoundError('Run the notebook from the example folder or repository.')
if str(EXAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLE_DIR))
import importlib
import csi_dataset
import csi_models
importlib.reload(csi_dataset)
importlib.reload(csi_models)
from csi_dataset import (MOTION_FEATURES, PHASE_FEATURES, SLOW_FEATURES,
    load_dataset, load_annotations, partition_dataset, select_features, quality_mask)
from csi_models import (compare_unsupervised, run_change_models, compare_classifiers,
    classification_metrics, save_bundle, predict_bundle)
get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})
print('Kernel:', sys.executable)
print('Example:', EXAMPLE_DIR)


Kernel: E:\Espressif\tools\python\v6.1\venv\Scripts\python.exe
Example: E:\IOT\esp-csi\drafts\modeling_and_analysis


## Data contract and evaluation protocol

`dataset_manifest.csv` lists raw captures with `session_id`, `capture_file`,
`split` (`unassigned/train/validation/test`), `reference`, room, placement, day,
and link. A capture appears only once; all links of a session share its split.
Fill unknown metadata before making generalization claims.

Optional `labels.csv` uses `session_id,start_s,end_s,person_count,activity,notes`.
Times are seconds from the first valid packet. Blank count means unknown.
Only windows whose full raw dependency interval fits in one label interval are
labeled. Occupancy transitions/partially labeled windows are excluded from
classification. `count_class` is 0/1/2/3, where 3 means **3+**, not exactly three.
`events.csv` contains `session_id,time_s,event,notes` for signal-change events.
Event recall requires a complete annotation protocol for the evaluated sessions.

Features are recomputed from raw CSI at two scales. Long features join backward
at their **availability time**, never using future windows. Bad sampling yields
unknown predictions, not empty-room predictions. Periodicity is optional.
The two-second gap limit and minimum packet count remain editable heuristics.

All preprocessing belongs inside each editable Pipeline and is fitted on
training data only. Explicit session splits are preferred. For larger datasets,
reserve a final test day, then compare candidates on development data. Never
randomly split overlapping windows. This notebook does not automatically choose
a model using the final test set.


In [2]:
# Edit these parameters and feature lists before running experiments.
MANIFEST_PATH = EXAMPLE_DIR / 'dataset_manifest.csv'
LABELS_PATH = EXAMPLE_DIR / 'labels.csv'
EVENTS_PATH = EXAMPLE_DIR / 'events.csv'
FEATURE_COLUMNS = list(MOTION_FEATURES)
# Optional ablations: FEATURE_COLUMNS += PHASE_FEATURES or SLOW_FEATURES
FAST_PARAMETERS = dict(window_seconds=5.0, hop_seconds=1.0, min_packets=12, max_gap_seconds=2.0)
SLOW_PARAMETERS = dict(window_seconds=30.0, hop_seconds=5.0, min_packets=12, max_gap_seconds=2.0)
GROUP_COLUMN = 'session_id'  # or day/placement_id after filling the manifest
USE_SLOW = any(name in SLOW_FEATURES for name in FEATURE_COLUMNS)
PURGE_SECONDS = SLOW_PARAMETERS['window_seconds'] if USE_SLOW else FAST_PARAMETERS['window_seconds']
RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = EXAMPLE_DIR / 'reports' / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = EXAMPLE_DIR / 'models' / RUN_ID
SAVE_MODELS = True

dataset = load_dataset(MANIFEST_PATH, LABELS_PATH, FAST_PARAMETERS, SLOW_PARAMETERS,
                       use_slow_features=USE_SLOW)
events = load_annotations(EVENTS_PATH, kind='events')
display(dataset.groupby(['session_id', 'split']).agg(
    windows=('quality_ok', 'size'), quality_windows=('quality_ok', 'sum'),
    labeled_windows=('presence', 'count')))
print('Predictive inputs:', FEATURE_COLUMNS)
print('Timestamps, metadata, labels, packet gaps/rates, and quality flags are excluded from X.')
print('Reports:', REPORT_DIR)


,,windows,quality_windows,labeled_windows
session_id,split,,,
csi_20261018_220301,unassigned,115,87,0


Predictive inputs: ['amplitude_mean', 'amplitude_std_median', 'amplitude_mad_median', 'amplitude_cv_median', 'amplitude_step_median', 'amplitude_change_rate_median', 'normalized_std_median', 'normalized_step_median', 'subcarrier_abs_correlation', 'pca1_variance_fraction']
Timestamps, metadata, labels, packet gaps/rates, and quality flags are excluded from X.
Reports: E:\IOT\esp-csi\drafts\modeling_and_analysis\reports\20261020T005940573255Z


In [3]:
# Supervised evaluation requires explicit independent sessions; no one-session fallback.
parts, split_mode = partition_dataset(dataset, group_column=GROUP_COLUMN,
    allow_chronological=False, purge_seconds=PURGE_SECONDS)
train, validation, final_test = (parts[name] for name in ('train', 'validation', 'test'))
TARGET = 'presence'  # 'count_class' = 0 / 1 / 2 / 3+ (ordinal categories)
CV_FOLDS = 3
print(split_mode)
print('TARGET:', TARGET)
display(pd.DataFrame([dict(split=k, windows=len(v), labeled=v[TARGET].notna().sum()) for k,v in parts.items()]))


no split assigned: edit dataset_manifest.csv
TARGET: presence


,split,windows,labeled
0,train,0,0
1,validation,0,0
2,test,0,0


## Editable classifier list and metric hooks

Add any sklearn-compatible classifier pipeline to this array. Each is cloned
for every grouped fold, and preprocessing never fits on holdout observations.
Dummy is the minimum benchmark. Logistic regression gives a linear baseline;
constrained random forests test nonlinear relationships. Replace the scaler,
add a PCA step, or create feature-family experiments here.

CV metrics are calculated on held-out session groups. Separate validation
metrics include macro-F1, balanced accuracy, binary precision/recall and average
precision when defined, ordinal class-distance, confusion matrices, and false
positive episodes per annotated empty-room hour. Session-bootstrap intervals
are reported only with at least three independent validation sessions.


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

CLASSIFIER_PIPELINES = [
    {'name': 'dummy_prior', 'pipeline': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DummyClassifier(strategy='prior')),
    ])},
    {'name': 'logistic', 'pipeline': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('variance', VarianceThreshold()), ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=42)),
    ])},
    {'name': 'random_forest', 'pipeline': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('variance', VarianceThreshold()),
        ('model', RandomForestClassifier(n_estimators=200, max_depth=5,
            min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ])},
]
# Context contains the fitted pipeline, fold_metrics, train_rows, target and,
# when labeled validation data exists, validation_rows/prediction/probabilities.
EXTRA_CLASSIFICATION_METRICS = {
    # 'validation_accuracy': lambda c: accuracy_score(c['validation_rows'][c['target']], c['prediction']),
}
display(pd.DataFrame([{'name': p['name'], 'pipeline': str(p['pipeline'])} for p in CLASSIFIER_PIPELINES]))


,name,pipeline
0,dummy_prior,"Pipeline(steps=[('imputer', SimpleImputer(stra..."
1,logistic,"Pipeline(steps=[('imputer', SimpleImputer(stra..."
2,random_forest,"Pipeline(steps=[('imputer', SimpleImputer(stra..."


In [5]:
classification_comparison, classification_results = compare_classifiers(
    CLASSIFIER_PIPELINES, train, validation, FEATURE_COLUMNS, target=TARGET,
    group_column=GROUP_COLUMN, n_splits=CV_FOLDS, extra_metrics=EXTRA_CLASSIFICATION_METRICS)
display(classification_comparison)
classification_comparison.to_csv(REPORT_DIR/f'{TARGET}_classification_metrics.csv', index=False)
if not classification_results:
    print('No models trained: annotate labels.csv and assign independent sessions to splits.')
    print('No occupancy accuracy is inferred from the unlabeled capture.')


,model,status,train_rows,validation_rows,reason
0,dummy_prior,skipped,0,0,Need labeled training data with >=2 classes an...
1,logistic,skipped,0,0,Need labeled training data with >=2 classes an...
2,random_forest,skipped,0,0,Need labeled training data with >=2 classes an...


No models trained: annotate labels.csv and assign independent sessions to splits.
No occupancy accuracy is inferred from the unlabeled capture.


## Validation diagnostics and feature importance

Set an optional confidence cutoff only using development data. Maximum class
probability is uncalibrated unless you explicitly add grouped probability
calibration; it is not a guarantee of correctness. The plotted rejected windows
are unknown rather than negative predictions. Raw metric comparisons above
remain unthresholded so the coverage tradeoff is visible.


In [6]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

INSPECT_CLASSIFIER = 'logistic'
MIN_CLASS_PROBABILITY = 0.0  # tune on validation only; 0 disables confidence rejection
result = classification_results.get(INSPECT_CLASSIFIER)
if result is not None and 'prediction' in result:
    val = result['validation_rows'].copy()
    prediction = result['prediction']
    probability = result['probabilities']
    keep = np.ones(len(val), dtype=bool)
    if probability is not None:
        keep = probability.max(axis=1) >= MIN_CLASS_PROBABILITY
    display(result['pipeline'])
    print(f'Confidence acceptance among quality-passing labeled windows: {keep.mean():.1%}')
    display(pd.DataFrame(result['class_report']).T)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
    ConfusionMatrixDisplay.from_predictions(val[TARGET].astype(int), prediction, ax=axes[0], colorbar=False)
    axes[0].set_title('Validation confusion (before confidence rejection)')
    for session, group in val.assign(prediction=prediction, accepted=keep).groupby('session_id'):
        axes[1].plot(group.window_end_s, group[TARGET], '-', label=f'{session}: actual')
        axes[1].scatter(group.window_end_s, np.where(group.accepted, group.prediction, np.nan),
                         s=18, label=f'{session}: predicted')
    axes[1].set(xlabel='Window end / availability (s)', ylabel=TARGET, title='Validation predictions')
    axes[1].legend(fontsize=8)
    plt.show()
    importance = permutation_importance(result['pipeline'], select_features(val, FEATURE_COLUMNS),
        val[TARGET].astype(int), scoring='f1_macro', n_repeats=5, random_state=42)
    display(pd.DataFrame({'feature': FEATURE_COLUMNS, 'importance': importance.importances_mean,
                         'std': importance.importances_std}).sort_values('importance', ascending=False))
    print('Permutation importance is descriptive on correlated validation windows; not a confidence interval.')
else:
    print('Validation diagnostics require a fitted model and labeled validation sessions.')


Validation diagnostics require a fitted model and labeled validation sessions.


## Save the chosen pipeline and final-test gate

Freeze the candidate and cutoff after development. Test prediction uses that
fitted pipeline without refitting. Changing the chosen pipeline after examining
test results turns the test set into development data.


In [7]:
SELECTED_CLASSIFIER = INSPECT_CLASSIFIER
RUN_FINAL_TEST = False
chosen = classification_results.get(SELECTED_CLASSIFIER)
if SAVE_MODELS and chosen:
    save_bundle(MODEL_DIR/f'{TARGET}_{SELECTED_CLASSIFIER}.joblib', chosen['pipeline'],
        FEATURE_COLUMNS, dict(target=TARGET, split_mode=split_mode,
        training_sessions=train.session_id.unique().tolist(),
        fast_parameters=FAST_PARAMETERS, slow_parameters=SLOW_PARAMETERS,
        confidence_cutoff=MIN_CLASS_PROBABILITY,
        manifest=MANIFEST_PATH.read_text(encoding='utf-8'),
        labels=LABELS_PATH.read_text(encoding='utf-8')))
if RUN_FINAL_TEST and chosen and len(final_test):
    output = final_test.copy()
    output['prediction'] = predict_bundle(dict(pipeline=chosen['pipeline'], features=FEATURE_COLUMNS), output)
    accepted = quality_mask(output, FEATURE_COLUMNS)
    if hasattr(chosen['pipeline'], 'predict_proba') and accepted.any():
        p = chosen['pipeline'].predict_proba(select_features(output.loc[accepted], FEATURE_COLUMNS))
        output.loc[accepted, 'max_class_probability'] = p.max(axis=1)
        output.loc[output.max_class_probability < MIN_CLASS_PROBABILITY, 'prediction'] = np.nan
    labeled = output[TARGET].notna() & output.prediction.notna() & output.label_status.eq('stable')
    if labeled.any():
        display(pd.DataFrame([classification_metrics(output.loc[labeled, TARGET].astype(int).to_numpy(),
            output.loc[labeled, 'prediction'].astype(int).to_numpy())]))
    print('Overall accepted test coverage:', output.prediction.notna().mean())
    output.to_csv(REPORT_DIR/f'{TARGET}_final_test_predictions.csv', index=False)
if not RUN_FINAL_TEST:
    print('Final test locked; no test scores calculated.')


Final test locked; no test scores calculated.
